# Demo — from `demo/config.yaml` to the final document

Input is a YAML file, not settings in this notebook — edit
`demo/config.yaml` (model, extractor, fallback, sample range) and
re-run this notebook top to bottom. Nothing here needs to change.

**How the PDF is read.** A normal PDF goes through `pdfplumber`, which
takes the text layer together with the bold, italic and underline cues
its fonts carry. A PDF whose text layer is missing or broken — a scanned
page, or fonts with no character-to-text mapping — fails a readability
check *before* the model sees it, and is re-read from its page images by
**LightOnOCR** instead (`fallback: auto`). The check is per document, so
clean PDFs never touch the fallback; when one does, `[fallback]` lines
below say which document and why.


## Input — demo/config.yaml, as written on disk


In [1]:
import os
from pathlib import Path

if Path.cwd().name == 'notebooks':
    os.chdir(Path.cwd().parent)

import json

import dmpbridge
print(f'dmpbridge {dmpbridge.__version__}, installed at {Path(dmpbridge.__file__).parent}')

from dmpbridge.core import paths as P
from dmpbridge.evaluation.experiment import ExperimentConfig, Experiment

CONFIG_PATH = Path('demo/config.yaml')
print(CONFIG_PATH.read_text(encoding='utf-8'))


dmpbridge 0.1.0, installed at C:\Users\Nahid\dmpbridge\dmpbridge
# Edit this file, then run:
#   python scripts/run_demo.py
#
# Results (the final labeled DMP document, one JSON per sample) are written
# into demo/output/ when it finishes.

name: Demo run
strategy: wholedoc
provider: ollama
host: http://localhost:11434

model: gemma4:e4b          # any model already pulled in Ollama

# How the PDF is read. pdfplumber handles a normal PDF with a real text layer;
# if that text comes out garbled (a scanned page, or fonts with no character
# mapping) the document is re-read from its page images by LightOnOCR instead.
# Per document — clean PDFs never touch the fallback.
extractor: pdfplumber       # pdfplumber | lightonocr | docling
fallback: auto              # auto = LightOnOCR 

pdf_dir: data/input/pdfs    # expects sample1.pdf, sample2.pdf, ...
sample_start: 12
sample_end: 12                # keep this small for a quick demo run



## Run it


In [2]:
cfg = ExperimentConfig.from_yaml(CONFIG_PATH)
exp = Experiment(cfg)
exp.run()

print(f'\n{cfg.name}: {len(cfg.models)} model(s), {len(cfg.extractors)} extractor(s), '
      f'samples {cfg.sample_start}-{cfg.sample_end}')



Demo run: 1 model(s), 1 extractor(s), samples 12-12


## Output — the final document

Same content `scripts/run_demo.py` copies into `demo/output/final/`; read here
directly from the standard pipeline location so this always reflects the latest run.


In [3]:
model, extractor = cfg.models[0], cfg.extractors[0]
tag = cfg.tag_for(model, extractor)

for n in cfg.sample_range:
    final = P.final_path(tag, n)
    if not final.exists():
        continue
    doc = json.loads(final.read_text(encoding='utf-8'))
    template = doc['narrative']['template']

    print(f'=== sample{n} ===')
    print(f'TITLE: {template["title"]}\n')
    for i, section in enumerate(template['section'], 1):
        print(f'{i}. {section["title"]}')
        for q in section['question']:
            answer = q['answer']['json']['answer']
            print(f'   Q: {q["text"][:70]}')
            print(f'   A: {answer[:90]}{"..." if len(answer) > 90 else ""}')
    print()


=== sample12 ===
TITLE: C-GEM: Plan for Data Management

1. Overview
2. Data Sharing Across Teams
   Q: Data Sharing Across Teams
   A: The C-GEM team will store, share, disseminate, and archive data following well-established...
3. 1. C-GEM Research Products
   Q: 1. C-GEM Research Products
   A: (A) Materials. C-GEM scientists will produce new chemical compounds, new biological reagen...
4. 2. Data Format
   Q: 2. Data Format
   A: Collected data will be formatted in several ways: (A) images as .tiff files; (B) experimen...
5. 3. Access to Data and Data Sharing Practices and Policies
   Q: 3. Access to Data and Data Sharing Practices and Policies
   A: (A) Intellectual Property, copyrights, and data will be administered in accordance with th...
6. 4. Policies for Re-use, Re-distribution, and Production of Derivatives
   Q: 4. Policies for Re-use, Re-distribution, and Production of Derivatives
   A: The Director has overall responsibility for both data management and dissemination. C-

## Save — copy the result into demo/output/

`exp.run()` writes to the pipeline's standard location under `data/output/`.
This copies each stage for the samples in the config into
`demo/output/{labeled,structured,final}/` — the same layout `scripts/run_demo.py` uses.


In [4]:
import shutil

OUTPUT_DIR = Path('demo/output')
STAGES = [('labeled', P.labeled_path), ('structured', P.structured_path), ('final', P.final_path)]

for n in cfg.sample_range:
    for stage, resolve in STAGES:
        src = resolve(tag, n)
        if not src.exists():
            continue
        dest = OUTPUT_DIR / stage / f'sample{n}.json'
        dest.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dest)
        print(f'{stage:10} -> {dest}')


labeled    -> demo\output\labeled\sample12.json
structured -> demo\output\structured\sample12.json
final      -> demo\output\final\sample12.json


## The labels, as a document

Every block of text with the label the model gave it, colour-coded:
title, section heading, section description, question, answer.


In [5]:
from html import escape
from itertools import groupby
from IPython.display import HTML, display

# label -> (colour, text colour); same palette as the pipeline diagram
COLORS = {
    'title':               ('#1E406E', 'white'),
    'section.title':       ('#0F766E', 'white'),
    'section.description': ('#94A3B8', 'black'),
    'question.text':       ('#B45309', 'white'),
    'answer.text':         ('#CBD5E1', 'black'),
}
SIZE   = {'title': '20px', 'section.title': '16px'}
BOLD   = {'title', 'section.title', 'question.text'}


def pill(label, extra=''):
    bg, fg = COLORS.get(label, ('#ddd', 'black'))
    return (f"<span style='background:{bg};color:{fg};font-size:11px;padding:2px 9px;"
            f"border-radius:10px;font-family:monospace'>{escape(label)}{extra}</span>")


def group_html(label, texts):
    """One cell for a run of consecutive blocks with the same label."""
    bg, _ = COLORS.get(label, ('#ddd', 'black'))
    body = ''.join(f"<div style='margin-top:6px;white-space:pre-wrap'>{escape(x)}</div>" for x in texts)
    return (f"<div style='border-left:6px solid {bg};background:{bg}18;padding:6px 12px;"
            f"margin:6px 0;font-family:system-ui,sans-serif;"
            f"font-size:{SIZE.get(label, '13px')};"
            f"font-weight:{'bold' if label in BOLD else 'normal'}'>"
            f"{pill(label, f' x{len(texts)}' if len(texts) > 1 else '')}{body}</div>")


for n in cfg.sample_range:
    blocks = json.loads(P.labeled_path(tag, n).read_text(encoding='utf-8'))
    counts = {}
    for b in blocks:
        counts[b['label']] = counts.get(b['label'], 0) + 1
    legend = ' '.join(pill(l, f' x{c}') for l, c in counts.items())
    # consecutive blocks with the same label share one cell
    groups = [(label, [b['text'] for b in run])
              for label, run in groupby(blocks, key=lambda b: b['label'])]
    display(HTML(f"<h3 style='font-family:system-ui,sans-serif'>sample{n} - "
                 f"{len(blocks)} labeled blocks in {len(groups)} cells</h3><p>{legend}</p>"
                 + ''.join(group_html(label, texts) for label, texts in groups)))
